# DressApp Eyes — Gemma 4 E2B Merge + INT4 Quantize → GGUF for on-device

End-to-end Colab notebook that takes either the **stock `google/gemma-4-E2B-it`** or your **fine-tuned LoRA adapter on top of it** all the way to a single **INT4 GGUF** + **F16 mmproj** pair sized to run in ~2–3 GB of RAM on a Raspberry Pi 5 (8 GB), a phone, or any other edge device that supports `llama.cpp`.

## What this notebook is built against (verified against the official model card, April 2026)

`google/gemma-4-E2B-it` · Apache 2.0 · Google DeepMind · downloads >3.3 M

| Spec                          | Value |
|-------------------------------|------------------------------------|
| Total parameters              | **5.1 B** (2.3 B "effective" after PLE compression) |
| Decoder layers                | **35** (hybrid: sliding-window 512 + global; final layer always global) |
| Vocabulary                    | 262 K |
| Context window                | **128 K tokens** |
| Modalities (in)               | Text · Image (variable aspect ratio + visual token budget 70/140/280/560/1120) · **Audio** (30 s ASR/AST) |
| Modalities (out)              | Text |
| Vision encoder                | ~150 M params |
| Audio encoder                 | ~300 M params |
| Architectural specials        | Per-Layer Embeddings (PLE), unified K/V on global layers, Proportional RoPE (p-RoPE) |
| Native HF class               | **`AutoModelForMultimodalLM`** (text-only: `AutoModelForCausalLM`) |
| Chat template                 | Native `system` / `user` / `assistant` roles; reasoning toggled by `<\|think\|>` in the system prompt |
| Recommended sampling          | `temperature=1.0  top_p=0.95  top_k=64` (we override to `temp=0.0` for JSON-strict generation) |

## What this notebook produces

```
/content/drive/MyDrive/DressApp_Gemma4_E2B_Training/
├── Eyes_v3_Gemma4_E2B/                # YOUR LoRA adapter        (optional input)
├── Eyes_v3_Gemma4_E2B_merged/         # merged HF checkpoint     (§3, only if adapter present)
└── Eyes_v3_Gemma4_E2B_gguf/           # GGUF outputs             (§4)
    ├── Eyes_v3_Gemma4_E2B-F16.gguf            # ~10 GB — re-quant source, delete after Q4 verified
    ├── Eyes_v3_Gemma4_E2B-mmproj-F16.gguf     # ~600 MB — vision tower (stays F16, per spec)
    └── Eyes_v3_Gemma4_E2B-Q4_K_M.gguf         # ~2-3 GB — on-device target
```

The `mmproj` (vision tower) stays in F16 — quantizing the ~150 M-param vision encoder gains <100 MB but historically wrecks JSON-schema adherence on garment images. The on-device RAM win comes almost entirely from the LM side, which is what Q4_K_M attacks.

## Mode of operation

The notebook **auto-detects** what you have on Drive:

* If `ADAPTER_DIR` exists and contains an `adapter_config.json` → **merge** stage (§3) runs and writes the merged HF checkpoint to `…_merged/`. The GGUF stage (§4) then reads from `…_merged/`.
* If `ADAPTER_DIR` is missing or empty → **skip merge**, GGUF stage reads `google/gemma-4-E2B-it` directly. Useful for benchmarking the stock model on your edge device before deciding whether to fine-tune.

> **Runtime requirements**
> - Colab Pro recommended (**L4 / A100 / V100, ≥24 GB GPU RAM**). T4 is OK for the merge stage but tight; the GGUF conversion + quantize step is CPU-bound and fine on any tier.
> - ~30 GB free space on Drive (the intermediate F16 GGUF is ~10 GB; delete it after Q4 passes smoke test in §5).
> - Read access to `google/gemma-4-E2B-it` on Hugging Face (accept the Apache-2 license once on the model page).


## §1 · Setup


In [ ]:
# Core stack. Gemma 4 needs Transformers with the gemma4 architecture registered.
# CRITICAL: do NOT --upgrade pillow or torchvision. Colab ships those pinned to
# its torch build; recent pillow 12.x has an internal _Ink import bug that
# torchvision triggers (see https://github.com/python-pillow/Pillow/issues).
# We install only what we genuinely need, and we pin pillow safely.
%pip install -q "transformers>=4.50.0" "peft>=0.13.0" "accelerate>=0.34" "safetensors>=0.4.5" "sentencepiece>=0.2.0" "pillow<12" librosa
%pip install -q huggingface_hub

# Sanity-check the Pillow import chain (catches the _Ink bug early instead of
# letting it blow up inside transformers).
import PIL
from PIL import Image, ImageDraw, ImageText  # noqa: F401 — this is the failing import on bad pillow
print(f"Pillow {PIL.__version__} OK")

# Build deps for llama.cpp
!apt-get -qq install -y cmake build-essential git ninja-build > /dev/null

import os, json, gc, time, textwrap, subprocess, shutil, pathlib, re
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(cpu)")


In [ ]:
# Hugging Face login — google/gemma-4-E2B-it is gated behind license acceptance.
from huggingface_hub import login
import getpass, os
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (read scope, gemma-4 license accepted): ")
login(HF_TOKEN)


## §2 · Drive paths & adapter auto-detection

Edit the three paths below once, everything else is derived.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Inputs ───────────────────────────────────────────────────────────
BASE_MODEL_ID   = "google/gemma-4-E2B-it"
# Point ADAPTER_DIR at your LoRA adapter on Drive. If it does not exist OR does
# not contain adapter_config.json, the notebook will skip §3 and quantize the
# stock base model instead.
ADAPTER_DIR     = "/content/drive/MyDrive/DressApp_Gemma4_E2B_Training/Eyes_v3_Gemma4_E2B"

# ── Outputs ──────────────────────────────────────────────────────────
ROOT_OUT        = "/content/drive/MyDrive/DressApp_Gemma4_E2B_Training"
MERGED_DIR      = f"{ROOT_OUT}/Eyes_v3_Gemma4_E2B_merged"
GGUF_DIR        = f"{ROOT_OUT}/Eyes_v3_Gemma4_E2B_gguf"
LLAMA_CPP_DIR   = "/content/llama.cpp"          # local Colab disk for the build artefacts

for d in (MERGED_DIR, GGUF_DIR):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# ── Adapter auto-detection + compatibility check ────────────────────────────
# We also check the adapter's `base_model_name_or_path` against BASE_MODEL_ID;
# if the adapter was trained on a different base (e.g. an older gemma-3n
# checkpoint), PEFT will crash or silently produce garbage on merge. Refuse
# to enter MERGE mode unless the bases match. User can override by setting
# FORCE_MERGE_DESPITE_BASE_MISMATCH = True.
FORCE_MERGE_DESPITE_BASE_MISMATCH = False

ADAPTER_EXISTS = (
    pathlib.Path(ADAPTER_DIR).exists()
    and pathlib.Path(f"{ADAPTER_DIR}/adapter_config.json").exists()
)
ADAPTER_BASE = None
ADAPTER_COMPATIBLE = False
if ADAPTER_EXISTS:
    cfg = json.loads(pathlib.Path(f"{ADAPTER_DIR}/adapter_config.json").read_text())
    ADAPTER_BASE = cfg.get("base_model_name_or_path", "<missing in adapter_config.json>")
    # normalise: HF ids are case-insensitive on the path side; compare lowercased
    base_lc, target_lc = str(ADAPTER_BASE).lower(), BASE_MODEL_ID.lower()
    ADAPTER_COMPATIBLE = (base_lc == target_lc) or (target_lc.split('/')[-1] in base_lc)

ADAPTER_PRESENT = ADAPTER_EXISTS and (ADAPTER_COMPATIBLE or FORCE_MERGE_DESPITE_BASE_MISMATCH)

if ADAPTER_PRESENT:
    SOURCE_FOR_GGUF = MERGED_DIR
    MODE = "MERGE+QUANTIZE"
elif ADAPTER_EXISTS and not ADAPTER_COMPATIBLE:
    SOURCE_FOR_GGUF = BASE_MODEL_ID
    MODE = "QUANTIZE-ONLY (adapter base mismatch)"
    print("=" * 72)
    print("  ADAPTER BASE MISMATCH DETECTED — SKIPPING MERGE")
    print("=" * 72)
    print(f"  Adapter was trained on: {ADAPTER_BASE}")
    print(f"  Target base is        : {BASE_MODEL_ID}")
    print("  These have incompatible architectures (different layer shapes,")
    print("  different module names). Merging would either crash with a")
    print("  shape-mismatch error or attach garbage weights.")
    print("")
    print("  §3 will be SKIPPED. §4 will quantize the stock base model.")
    print("")
    print("  If you really want to force the merge anyway, set")
    print("      FORCE_MERGE_DESPITE_BASE_MISMATCH = True")
    print("  above this cell and re-run. (Not recommended.)")
    print("=" * 72 + "\n")
else:
    SOURCE_FOR_GGUF = BASE_MODEL_ID
    MODE = "QUANTIZE-ONLY (no adapter found)"
    print(f"NOTE: no LoRA adapter at {ADAPTER_DIR!r} — §3 (merge) will be skipped.")
    print("      The GGUF in §4 will be built from the stock base model.\n")

print(f"Mode               : {MODE}")
print(f"Base model         : {BASE_MODEL_ID}")
print(f"Adapter source     : {ADAPTER_DIR}  (exists={ADAPTER_EXISTS}, compatible={ADAPTER_COMPATIBLE})")
if ADAPTER_BASE:
    print(f"Adapter trained on : {ADAPTER_BASE}")
print(f"Merged target      : {MERGED_DIR}")
print(f"GGUF target        : {GGUF_DIR}")
print(f"Source for GGUF    : {SOURCE_FOR_GGUF}")


## §3 · (Conditional) Merge the LoRA adapter into the base in bf16

This section runs **only** if `ADAPTER_PRESENT == True`. Otherwise the cells short-circuit and §4 reads the stock HF model directly.

Gemma 4 changes from Gemma 3n:
* Loader is **`AutoModelForMultimodalLM`** (not `AutoModelForImageTextToText`) because audio is now a first-class input.
* The chat template uses native `system` / `user` / `assistant` roles — no `<start_of_turn>` ceremony.
* No `<\|think\|>` token in the system prompt → reasoning is off (right choice for JSON-strict garment description).


In [ ]:
if not ADAPTER_PRESENT:
    print(f"SKIPPED — no adapter at {ADAPTER_DIR}. Jumping to §3b.")
else:
    from transformers import AutoModelForMultimodalLM, AutoProcessor
    from peft import PeftModel
    import torch.nn as nn

    print(f">>> loading base {BASE_MODEL_ID} in bfloat16 ...")
    t0 = time.time()
    base = AutoModelForMultimodalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    proc = AutoProcessor.from_pretrained(BASE_MODEL_ID)
    print(f"    base ready in {time.time()-t0:.1f}s")

    # ── PEFT compatibility shim for Gemma 4's custom Linear subclasses ─────
    # Gemma 4 wraps every `_proj` in `Gemma4ClippableLinear` (a thin wrapper
    # holding per-layer activation-clip thresholds for quant-aware deployment).
    # PEFT's dispatch table only knows about plain nn.Linear, so it raises
    # `ValueError: Target module Gemma4ClippableLinear(...) is not supported`.
    # We temporarily swap each wrapper's place in its parent with the inner
    # nn.Linear (which IS in PEFT's table), let PEFT attach + merge, then
    # re-wrap so the saved checkpoint preserves the original state-dict keys.
    WRAPPER_CLASSES = ("Gemma4ClippableLinear",)  # extend if more show up
    unwrapped = {}   # full_name -> (parent_module, attr_name, original_wrapper)

    def _walk(prefix, module):
        for name, child in list(module.named_children()):
            full = f"{prefix}.{name}" if prefix else name
            if child.__class__.__name__ in WRAPPER_CLASSES:
                if hasattr(child, "linear") and isinstance(child.linear, nn.Linear):
                    unwrapped[full] = (module, name, child)
                    setattr(module, name, child.linear)
                else:
                    print(f"  WARN: {full} is {child.__class__.__name__} but has no .linear submodule — leaving untouched")
            else:
                _walk(full, child)
    _walk("", base)
    print(f"    unwrapped {len(unwrapped)} {WRAPPER_CLASSES[0]} modules for PEFT compatibility")
    if len(unwrapped) == 0:
        print("    (none found — PEFT may handle Gemma 4 natively; proceeding)")

    print(f">>> attaching adapter from {ADAPTER_DIR} ...")
    peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR, torch_dtype=torch.bfloat16)

    print(">>> merging LoRA into base (in place) ...")
    t0 = time.time()
    merged = peft_model.merge_and_unload(progressbar=True)
    print(f"    merge done in {time.time()-t0:.1f}s")

    # Re-wrap: put each ClippableLinear back around its now-merged inner Linear.
    # This restores the original state-dict keys (...q_proj.linear.weight rather
    # than ...q_proj.weight) so save_pretrained writes a self-consistent checkpoint.
    for full, (parent, name, wrapper) in unwrapped.items():
        merged_inner = getattr(parent, name)            # post-merge nn.Linear
        wrapper.linear = merged_inner                   # reattach merged weights
        setattr(parent, name, wrapper)                  # restore wrapper at the original slot
    print(f"    re-wrapped {len(unwrapped)} modules — state-dict keys restored")

    remaining = [n for n, _ in merged.named_parameters() if "lora_" in n]
    assert not remaining, f"merge_and_unload left {len(remaining)} lora_ params — check PEFT version"
    print("    OK: no residual lora_* params")


In [ ]:
if not ADAPTER_PRESENT:
    print("SKIPPED — no merge produced, nothing to sanity-check.")
else:
    # Quick "did the merge actually do anything" check — run one inference
    # against a blank canvas. With the Eyes prompt the model should still
    # emit the 18-field JSON schema after the merge.
    from PIL import Image
    blank = Image.new("RGB", (448, 448), (240, 240, 240))
    messages = [
        {"role": "system", "content": "You are DressApp Eyes. Return JSON only, no prose."},
        {"role": "user",   "content": [
            {"type": "image", "image": blank},
            {"type": "text",  "text": "Describe this garment in the DressApp 18-field schema."},
        ]},
    ]
    inputs = proc.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(merged.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = merged.generate(**inputs, max_new_tokens=384, do_sample=False, temperature=0.0)
    response = proc.decode(out[0][input_len:], skip_special_tokens=False)
    parsed   = proc.parse_response(response) if hasattr(proc, "parse_response") else response
    print("=== blank-canvas response (post-merge, pre-quantization) ===")
    print(str(parsed)[:1000])


In [ ]:
if not ADAPTER_PRESENT:
    print("SKIPPED — no merge produced, nothing to save.")
else:
    print(f">>> saving merged HF model -> {MERGED_DIR}")
    t0 = time.time()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="4GB")
    proc.save_pretrained(MERGED_DIR)
    print(f"    saved in {time.time()-t0:.1f}s")

    # Free GPU memory before §4 spawns llama.cpp
    del merged, peft_model, base, proc
    gc.collect()
    torch.cuda.empty_cache()

    !ls -lh "$MERGED_DIR"


## §3b · Discover the live module tree → compile the FP16 keep-list

Your previous Gemma-3n module audit is **architecturally invalid** for Gemma 4 (different layer counts, renamed sub-blocks, audio is now native). Instead of hard-coding paths that might not exist, this cell loads the **state-dict keys only** (no weights into RAM), groups them into the four protected families, and prints a summary so you can spot anything unexpected.

The four families that stay in F16 regardless of quant level:

1. **Vision tower** — every linear in the ~150 M-param vision encoder + the patch-embedder + pooler.
2. **Audio tower** — every linear in the ~300 M-param audio encoder. Gemma 4 E2B is genuinely trimodal; quantizing audio has historically broken vision residual paths even when you do not feed audio in.
3. **Cross-modal embed projections** — `embed_vision.*` and `embed_audio.*` (the adapters that splice non-text token streams into the LM token space).
4. **PLE family** — every per-layer-embedding lookup table. Gemma 4 keeps PLE from 3n: each decoder layer has its own small embedding for every token, summed into the hidden state. Quantizing these tiny tables visibly degrades JSON-schema adherence.

Plus standard practice: **token embedding** (F16) and **every norm** (F32).


In [ ]:
from transformers import AutoModelForMultimodalLM
import torch

# Stream the safetensors index instead of materialising weights — saves ~12 GB RAM.
# (HF >=4.50 supports config-only loading via from_pretrained(..., state_dict={}) but
# the universally portable way is to grab the index manually.)
import json, urllib.request
from huggingface_hub import hf_hub_download, list_repo_files

SOURCE = SOURCE_FOR_GGUF if ADAPTER_PRESENT else BASE_MODEL_ID

if pathlib.Path(SOURCE).exists():
    # Local merged checkpoint
    idx_path = pathlib.Path(SOURCE) / "model.safetensors.index.json"
    state_keys = sorted(json.loads(idx_path.read_text())["weight_map"].keys()) \
        if idx_path.exists() \
        else sorted(__import__("safetensors").safe_open(
            next(pathlib.Path(SOURCE).glob("*.safetensors")), framework="pt").keys())
else:
    # Stock HF repo — pull only the index file
    idx_local = hf_hub_download(SOURCE, "model.safetensors.index.json")
    state_keys = sorted(json.loads(open(idx_local).read())["weight_map"].keys())

print(f"discovered {len(state_keys)} tensors in {SOURCE}")

# Group keys into families. Patterns are anchored against the HF state-dict prefix
# (which for Gemma 4 wraps the model under either `model.` or directly at root depending
# on the HF version) — we therefore match both with the optional `^(model\.)?` prefix.
FAMILY_PATTERNS = {
    "vision_tower"      : r"^(model\.)?vision_tower(\.|$)",
    "audio_tower"       : r"^(model\.)?audio_tower(\.|$)",
    "embed_vision"      : r"^(model\.)?embed_vision(\.|$)",
    "embed_audio"       : r"^(model\.)?embed_audio(\.|$)",
    "per_layer_embed"   : r"^(model\.)?language_model\.embed_tokens_per_layer(\.|$)",
    "per_layer_proj"    : r"^(model\.)?language_model\.layers\.\d+\.per_layer_projection(\.|$)",
    "per_layer_model"   : r"^(model\.)?language_model\.per_layer_model_projection(\.|$)",
    "per_layer_norm"    : r"^(model\.)?language_model\.per_layer_projection_norm(\.|$)",
    "token_embedding"   : r"^(model\.)?language_model\.embed_tokens(\.|$)",
    "lm_head"           : r"^lm_head(\.|$)",
    "norms"             : r"(_norm|\.norm)(\.|$)",
}

import collections
fam_counts = collections.OrderedDict((f, 0) for f in FAMILY_PATTERNS)
fam_examples = collections.OrderedDict((f, []) for f in FAMILY_PATTERNS)
fam_compiled = {f: re.compile(p) for f, p in FAMILY_PATTERNS.items()}
unmatched = []

for k in state_keys:
    matched = False
    for fam, rx in fam_compiled.items():
        if rx.search(k):
            fam_counts[fam] += 1
            if len(fam_examples[fam]) < 2:
                fam_examples[fam].append(k)
            matched = True
            break
    if not matched:
        unmatched.append(k)

print("\n=== FP16 keep-list coverage on the live Gemma 4 graph ===")
for fam, n in fam_counts.items():
    print(f"  {fam:<18}  {n:>4}  e.g. {fam_examples[fam][:1]}")
print(f"\n  (LM linears to be Q4-quantized, i.e. unmatched here): {len(unmatched)}")
print(f"  sample LM linears: {unmatched[:3]}")

if fam_counts["vision_tower"] == 0:
    print("\n  WARNING: vision_tower matched 0 tensors. The state-dict prefix may differ on")
    print("  your Transformers version — inspect `state_keys[:20]` and adjust FAMILY_PATTERNS.")
if fam_counts["audio_tower"] == 0:
    print("\n  WARNING: audio_tower matched 0 tensors. If you stripped audio from your custom")
    print("  fine-tune this is expected; otherwise inspect state_keys.")

# Combined keep-list, exported for §4 to map onto GGUF tensor names.
KEEP_FP16_HF_REGEX = list(FAMILY_PATTERNS.values())
KEEP_FP16 = [re.compile(p) for p in KEEP_FP16_HF_REGEX]
def should_keep_fp16(name): return any(rx.search(name) for rx in KEEP_FP16)
print("\n  keep-list compiled —", sum(1 for k in state_keys if should_keep_fp16(k)),
      "of", len(state_keys), "tensors will be FP16/F32 in the final GGUF.")


## §4 · GGUF export via `llama.cpp` — INT4 (Q4_K_M) on-device target

### 4a · Build `llama.cpp` from master

> **Gemma 4 freshness check.** Gemma 4 was released in April 2026; baseline `master` should have the `LLM_ARCH_GEMMA4` symbol. If `convert_hf_to_gguf.py` errors with *"unknown architecture: gemma4"*, see §6a (build from the `gg/gemma-4` feature branch).


In [ ]:
# Build llama.cpp from master with CUDA, surfacing all logs on failure.
#
# CRITICAL: we do NOT `pip install -r requirements.txt` from llama.cpp.
# That file pins torch==2.6.0+cpu (because the convert script only needs
# CPU tensor inspection), which silently downgrades Colab's CUDA torch
# and breaks every other cell in this notebook. Instead we install ONLY
# the specific packages convert_hf_to_gguf.py actually imports.
import os, sys, pathlib, subprocess, shutil

if not pathlib.Path(LLAMA_CPP_DIR).exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git "$LLAMA_CPP_DIR"

os.chdir(LLAMA_CPP_DIR)

# Surgical deps — skip the torch-downgrading requirements.txt entirely.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gguf", "sentencepiece", "protobuf", "mistral-common"],
               check=False)

def run_and_log(cmd, log_path, label):
    """Run cmd, redirect output to log_path, print tail on failure."""
    print(f">>> {label} ...")
    with open(log_path, "w") as logf:
        rc = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print(f"FAILED ({label}, rc={rc}) — last 60 lines of {log_path}:")
        with open(log_path) as f:
            tail = f.readlines()[-60:]
        print("".join(tail))
        raise SystemExit(f"{label} failed; see {log_path} for the full log.")
    print(f"    OK")

# 1) configure  (drop -DGGML_CUDA=ON if the CUDA toolkit version mismatches;
#                CPU-only build still works fine for quantize + small smoke tests)
USE_CUDA = subprocess.run(["which", "nvcc"], capture_output=True).returncode == 0
cmake_args = ["cmake", "-B", "build",
              "-DLLAMA_CURL=OFF",
              "-DCMAKE_BUILD_TYPE=Release",
              "-G", "Ninja"]
if USE_CUDA:
    cmake_args.append("-DGGML_CUDA=ON")
    print("nvcc found — building with CUDA")
else:
    print("nvcc NOT found — building CPU-only (quantize works fine, mtmd-cli will use CPU)")
run_and_log(cmake_args, "/tmp/cmake.log", "cmake configure")

# 2) build
nproc = str(os.cpu_count() or 4)
build_args = ["cmake", "--build", "build", "--config", "Release",
              "-j", nproc,
              "--target", "llama-quantize", "llama-mtmd-cli", "llama-cli"]
run_and_log(build_args, "/tmp/build.log", "cmake build (this takes 4-8 minutes)")

# 3) verify
for binname in ["llama-quantize", "llama-mtmd-cli", "llama-cli"]:
    p = pathlib.Path(LLAMA_CPP_DIR) / "build" / "bin" / binname
    assert p.exists(), f"{binname} did not get built — inspect /tmp/build.log"
    print(f"  {p}  ({p.stat().st_size/1e6:.1f} MB)")
print("\nllama.cpp ready.")


### 4b · Convert HF → F16 GGUF (LM) + F16 mmproj (vision + cross-modal)

`convert_hf_to_gguf.py --mmproj` writes two files: the language model and the vision-tower / cross-modal projection. For Gemma 4 the audio tower stays inside the LM-side GGUF (llama.cpp does not yet have an "mmproj-audio" slot in its multimodal CLI) — but since we keep every audio-tower tensor in F16 anyway via §4c overrides, this has zero quality impact.


In [ ]:
F16_LM     = f"{GGUF_DIR}/Eyes_v3_Gemma4_E2B-F16.gguf"
F16_MMPROJ = f"{GGUF_DIR}/Eyes_v3_Gemma4_E2B-mmproj-F16.gguf"

# Modern llama.cpp convert_hf_to_gguf.py: --mmproj means 'convert VISION TOWER
# ONLY as an mmproj' (NOT 'convert LM and also write mmproj sibling'). So we
# need TWO conversion passes — one for the mmproj, one for the LM.
import subprocess, time
os.chdir(LLAMA_CPP_DIR)

print(">>> pass 1/2: vision tower -> mmproj F16 ...")
t0 = time.time()
r1 = subprocess.run([
    "python", "convert_hf_to_gguf.py", SOURCE_FOR_GGUF,
    "--outtype", "f16", "--outfile", F16_MMPROJ, "--mmproj",
], capture_output=True, text=True, timeout=900)
print(r1.stdout[-1500:])
if r1.returncode != 0:
    print("STDERR:", r1.stderr[-1500:]); raise SystemExit("mmproj conversion failed")
print(f"    mmproj done in {time.time()-t0:.1f}s")

print("\n>>> pass 2/2: language model -> LM F16 ...")
t0 = time.time()
r2 = subprocess.run([
    "python", "convert_hf_to_gguf.py", SOURCE_FOR_GGUF,
    "--outtype", "f16", "--outfile", F16_LM,
], capture_output=True, text=True, timeout=900)
print(r2.stdout[-1500:])
if r2.returncode != 0:
    print("STDERR:", r2.stderr[-1500:]); raise SystemExit("LM conversion failed")
print(f"    LM done in {time.time()-t0:.1f}s")

print("\nIntermediate F16 artefacts (delete after §5 passes):")
!ls -lh "$F16_LM" "$F16_MMPROJ"


### 4c · Quantize the LM to Q4_K_M with FP16 overrides

`llama-quantize --tensor-type <regex>=<TYPE>` overrides the global quant level on a per-tensor basis. The patterns here are written against the **GGUF tensor names** that `convert_hf_to_gguf.py` emits, which differ from HF names:

| HF state-dict key                                           | GGUF tensor name           |
|-------------------------------------------------------------|----------------------------|
| `model.language_model.embed_tokens`                         | `token_embd`               |
| `model.language_model.layers.N.self_attn.{q,k,v,o}_proj`    | `blk.N.attn_{q,k,v,output}`|
| `model.language_model.layers.N.mlp.{gate,up,down}_proj`     | `blk.N.ffn_{gate,up,down}` |
| `model.language_model.layers.N.per_layer_projection`        | `blk.N.per_layer_proj`     |
| `model.language_model.embed_tokens_per_layer`               | `per_layer_token_embd`     |
| `model.language_model.per_layer_model_projection`           | `per_layer_model_proj`     |
| `model.language_model.per_layer_projection_norm`            | `per_layer_proj_norm`      |
| `*.norm.*` / `*_norm.*` / `q_norm`, `k_norm`, `v_norm`      | preserved suffix on `blk.*`|

Vision / audio / `embed_vision` / `embed_audio` land in the **mmproj** file in §4b — they are not in the LM GGUF, so the LM-side keep-list collapses to: `token_embd`, every `per_layer_*` family, and every norm.


In [ ]:
GGUF_KEEP_FP16_OVERRIDES = [
    # ── Token embedding: handled below via --token-embedding-type=Q8_0 ──
    #    (tied with lm_head; F16 wastes ~600 MB, Q8_0 is the safe step on small VLMs)
    #
    # ── PLE family ──
    #   `per_layer_token_embd`: the giant ~4.7 GB lookup table.  NOT protected:
    #   PLE paper §3.2 designed lookup tables to tolerate aggressive quant.
    #   The values are added to hidden state (not matmul-cascaded), so quant
    #   noise gets absorbed.  Default Q4_K_M cuts this from 4.7 GB → ~1.2 GB,
    #   the single largest size win on a Gemma 4 E2B GGUF.
    #
    #   The smaller PLE projection + per-block per_layer_proj stay F16 —
    #   these are matmul ops in the residual path, not lookups (~80 MB total).
    r"^per_layer_model_proj\..*=F16",
    r"^blk\.\d+\.per_layer_proj\..*=F16",

    # ── Per-layer residual scalars (Gemma 4 specific, 1 float per LM layer) ──
    r"^blk\.\d+\.layer_scalar.*=F16",

    # ── Norms (every variety in F32) ──
    r".*\.norm\..*=F32",
    r".*_norm\..*=F32",
    r".*\.layernorm\..*=F32",
    r".*\.attn_(q|k|v)_norm\..*=F32",
    r".*\.post_per_layer_input_norm\..*=F32",
    r"^per_layer_proj_norm\..*=F32",
]

def quantize(target_quant, out_name):
    out_path = f"{GGUF_DIR}/{out_name}"
    cmd = [f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"]
    for pat in GGUF_KEEP_FP16_OVERRIDES:
        cmd += ["--tensor-type", pat]
    # Tied embeddings: Q8_0 is the standard safe step on small VLMs.
    # F16 wastes ~600 MB; Q4 risks output-token quality on a 2.3B-effective model.
    cmd += ["--output-tensor-type",   "Q8_0"]
    cmd += ["--token-embedding-type", "Q8_0"]
    cmd += [F16_LM, out_path, target_quant]

    print(">>>", " ".join(cmd[:6]), "...", target_quant, "->", out_name)
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-2000:])
        raise RuntimeError(f"llama-quantize failed for {target_quant}")
    return out_path

P_Q4 = quantize("Q4_K_M", "Eyes_v3_Gemma4_E2B-Q4_K_M.gguf")

print("\n=== final on-device artefact pair ===")
!ls -lh "$GGUF_DIR/Eyes_v3_Gemma4_E2B-Q4_K_M.gguf" "$F16_MMPROJ"
print("\nTotal on-device footprint:")
!du -ch "$GGUF_DIR/Eyes_v3_Gemma4_E2B-Q4_K_M.gguf" "$F16_MMPROJ" | tail -1


## §5 · Smoke test: real garment image through `llama-mtmd-cli`

If this cell emits the 18-field JSON schema cleanly with `temperature=0.0`, your Q4 is shippable. If it skips fields or hallucinates structure, jump to §6b.


In [ ]:
TEST_IMAGE = "/content/garment.jpg"

# Trigger the upload prompt if the file is missing OR is a zero-byte stub.
# (Colab's /content/ is ephemeral — wiped on every runtime restart.)
import os
need_upload = (not pathlib.Path(TEST_IMAGE).exists()) or os.path.getsize(TEST_IMAGE) == 0
if need_upload:
    from google.colab import files
    print("Upload a garment photo (jpg/png) — it will be saved to /content/garment.jpg")
    up = files.upload()
    src = next(iter(up.keys()))
    shutil.copy(src, TEST_IMAGE)
print(f"Using image: {TEST_IMAGE}  ({os.path.getsize(TEST_IMAGE)/1024:.0f} KB)")

# Gemma 4 chat template: native system / user / assistant roles, image BEFORE text
# (per "Best Practice §4 · Modality order" in the model card).
# We DO NOT include <|think|> in the system prompt — reasoning off = strict JSON.
PROMPT = (
    "You are DressApp Eyes. Look at the garment in the image and return ONLY a "
    "JSON object with these 18 fields: category, subcategory, primary_color, "
    "secondary_colors, pattern, material, neckline, sleeve_length, length, fit, "
    "season, occasion, style, brand_visible, condition, has_text, notable_features, "
    "confidence. No prose, no markdown, no thinking, no <|think|>."
)

cmd = [
    f"{LLAMA_CPP_DIR}/build/bin/llama-mtmd-cli",
    "-m",        P_Q4,
    "--mmproj",  F16_MMPROJ,
    "--image",   TEST_IMAGE,
    "-p",        PROMPT,
    "-n",        "512",
    "--temp",    "0.0",     # JSON-strict; override the model-card 1.0 sampling
    "-ngl",      "99",       # offload to GPU on Colab; on Pi/phone llama.cpp picks automatically
    "--jinja",               # REQUIRED for Gemma 4 — its chat template uses advanced Jinja constructs
                              # (native system role, <|think|> token, etc.) that the fast-path renderer
                              # can't handle. Without this you get: 'this custom template is not supported'.
    "--no-warmup",           # skip the empty-run warmup that also hits the template renderer
    # NOTE: --chat-template-kwargs '{"enable_thinking": false}' would be the
    # ideal way to suppress Gemma 4's <|channel>thought trace, but that flag
    # has not landed in llama-mtmd-cli yet (only in llama-cli). The
    # parse_eyes_response() helper below strips the trace at parse time,
    # which is what production code should do anyway.
]
t0 = time.time()
res = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
dt = time.time() - t0
print(res.stdout[-3000:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-1500:])
print(f"\nelapsed: {dt:.1f}s   ({pathlib.Path(P_Q4).stat().st_size/1e9:.2f} GB LM + {pathlib.Path(F16_MMPROJ).stat().st_size/1e6:.0f} MB mmproj)")

# ── Production-side response parser ─────────────────────────────────
# Drop this into backend/app/services/garment_vision.py once you ship the GGUF.
# Strips Gemma 4's <|channel>thought ... <channel|> trace (if --chat-template-kwargs
# was ignored by your build) and parses the trailing JSON. Robust to leading prose.
import json, re
def parse_eyes_response(raw: str) -> dict:
    """Production parser for Eyes v3 (Gemma 4 E2B) GGUF responses."""
    if "<channel|>" in raw:
        raw = raw.split("<channel|>", 1)[1]   # drop the thinking trace
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        raise ValueError("no JSON object in response")
    return json.loads(m.group(0))

# Sanity-test the parser on this smoke test's output
try:
    parsed = parse_eyes_response(res.stdout)
    print("\n=== parsed JSON ({} fields) ===".format(len(parsed)))
    print(json.dumps(parsed, indent=2)[:1500])
except Exception as e:
    print(f"\nparser failed: {e}")


## §6 · Troubleshooting & on-device deployment notes

### 6a · `convert_hf_to_gguf.py` errors with *unknown architecture: gemma4*

Your `llama.cpp` clone predates Gemma 4 support. Update to the latest master:

```bash
cd /content/llama.cpp && git fetch --all && git checkout master && git pull \
  && pip install -r requirements.txt \
  && cmake --build build -j$(nproc)
```

If master still does not recognise it, build from the upstream PR branch (replace the branch name with whatever PR is open at the time — search the llama.cpp issue tracker for "gemma 4"):

```bash
cd /content/llama.cpp && git fetch origin pull/<PR_NUMBER>/head:gemma4 && git checkout gemma4 \
  && pip install -r requirements.txt && cmake --build build -j$(nproc)
```

### 6b · Q4_K_M regresses on the 18-field schema

Symptoms: JSON parses but drops fields, or the model starts emitting prose before the JSON, or `parse_response` returns an empty thought block.

Fix ladder — re-run §4c with one extra override at a time, stopping when the smoke test passes:

1. **Pin block 0 and block 34** (first and last LM blocks — empirically the most quant-sensitive):
   ```python
   GGUF_KEEP_FP16_OVERRIDES.append(r"^blk\.(0|34)\..*=F16")
   ```
2. **Pin every global-attention layer**. Gemma 4 alternates sliding-window and global attention with the final layer always global. The global layers carry the heavy lifting for long-range vision/audio token reasoning and tolerate quantization worst. If you can read `config.json` from the merged checkpoint, list the global indices; if not, conservatively pin every 6th layer starting from the last:
   ```python
   GGUF_KEEP_FP16_OVERRIDES.append(r"^blk\.(34|28|22|16|10|4)\..*=F16")
   ```
3. **Step up to Q5_K_M as the LM base** (changes size budget from ~2-3 GB to ~3-4 GB — still phone-friendly):
   ```python
   P_Q5 = quantize("Q5_K_M", "Eyes_v3_Gemma4_E2B-Q5_K_M.gguf")
   ```

### 6c · Running on a Raspberry Pi 5 (8 GB) or phone

The Q4_K_M + mmproj-F16 pair (~3 GB total) fits comfortably on Pi 5 and any phone with ≥ 4 GB RAM.

**Pi 5 (Raspberry Pi OS 64-bit):**
```bash
sudo apt install -y cmake build-essential
git clone --depth 1 https://github.com/ggml-org/llama.cpp && cd llama.cpp
cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build -j4 --target llama-mtmd-cli

./build/bin/llama-mtmd-cli \
   -m /path/Eyes_v3_Gemma4_E2B-Q4_K_M.gguf \
   --mmproj /path/Eyes_v3_Gemma4_E2B-mmproj-F16.gguf \
   --image  garment.jpg \
   -p       "..." \
   --temp   0.0 \
   -t       4              # all 4 Pi cores
```
Expected throughput: ~3-6 tok/s on Pi 5; first-token latency dominated by the image-encode pass (~3-5 s for a 1024-px garment).

**Android (Termux + llama.cpp):** same as Pi but built with `cmake -B build -DLLAMA_OPENMP=ON`. Or use `MediaPipe LlmInference` if you also export to LiteRT (out of scope here — we deliberately stuck to GGUF-only).

**iOS:** wrap `llama.cpp` in a Swift package via `llama-cpp-ios` or call into it from Capacitor. `llama-mtmd-cli`-equivalent multimodal inference on iOS still goes through llama.cpp's C API directly; the mmproj file is loaded the same way.

### 6d · Wiring into DressApp's `garment_vision.py`

When Q4 quality is verified on real data:

1. Copy the artefact pair to the production VPS:
   ```
   /var/models/eyes_v3/Eyes_v3_Gemma4_E2B-Q4_K_M.gguf
   /var/models/eyes_v3/Eyes_v3_Gemma4_E2B-mmproj-F16.gguf
   ```
2. Mirror the existing `custom_eyes_v1` block in `backend/app/services/garment_vision.py`, adding a `custom_eyes_v3_q4km_gemma4` provider that loads via `llama_cpp.Llama(model_path=…, mmproj_path=…)`.
3. Flip `config.eyes_provider.active` in MongoDB from `gemini` to `custom_eyes_v3_q4km_gemma4`.
4. Watch `/api/v1/admin/eyes/diagnostics` for 24 h before turning Gemini off.

### 6e · Once Q4 is verified, reclaim Drive space

```bash
rm "$GGUF_DIR/Eyes_v3_Gemma4_E2B-F16.gguf"   # ~10 GB intermediate, no longer needed
```

The shipping pair is just `Eyes_v3_Gemma4_E2B-Q4_K_M.gguf` + `Eyes_v3_Gemma4_E2B-mmproj-F16.gguf` (~3 GB total).
